In [52]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [53]:
df = pd.read_csv('rfm_segments.csv')
df_sales = pd.read_csv('online_retail_sales_cleaned.csv')
df_sales['InvoiceDate'] = pd.to_datetime(df_sales['InvoiceDate'])

In [54]:
df.head()

,Customer ID,Recency,Frequency,Monetary,Cluster,Segment
0,12346,326,12,77556.46,1,Regular
1,12347,2,8,5633.32,1,Regular
2,12348,75,5,2019.40,1,Regular
3,12349,19,4,4428.69,1,Regular
4,12350,310,1,334.40,0,At Risk


In [55]:
df.sort_values(by='Recency', ascending=True)

,Customer ID,Recency,Frequency,Monetary,Cluster,Segment
5178,17581,1,43,18757.75,1,Regular
4846,17243,1,69,20889.86,1,Regular
76,12423,1,10,2622.39,1,Regular
1161,13521,1,3,1093.65,1,Regular
1413,13777,1,61,56478.42,2,VIP
...,...,...,...,...,...,...
3454,15833,738,1,80.40,0,At Risk
5410,17818,738,1,130.18,0,At Risk
4660,17056,738,1,128.60,0,At Risk
5189,17592,739,1,148.30,0,At Risk


In [56]:
df.head(10)

,Customer ID,Recency,Frequency,Monetary,Cluster,Segment
0,12346,326,12,77556.46,1,Regular
1,12347,2,8,5633.32,1,Regular
2,12348,75,5,2019.40,1,Regular
3,12349,19,4,4428.69,1,Regular
4,12350,310,1,334.40,0,At Risk
5,12351,375,1,300.93,0,At Risk
6,12352,36,10,2849.84,1,Regular
7,12353,204,2,406.76,1,Regular
8,12354,232,1,1079.40,1,Regular
9,12355,214,2,947.61,1,Regular


In [57]:
df['Churned'] = (df['Recency'] > 90).astype(int)

In [58]:
df.head()

,Customer ID,Recency,Frequency,Monetary,Cluster,Segment,Churned
0,12346,326,12,77556.46,1,Regular,1
1,12347,2,8,5633.32,1,Regular,0
2,12348,75,5,2019.40,1,Regular,0
3,12349,19,4,4428.69,1,Regular,0
4,12350,310,1,334.40,0,At Risk,1


In [59]:
df['Churned'].value_counts(normalize=True) * 100

,proportion
Churned,
1,50.850629
0,49.149371


In [60]:
X = df[['Frequency', 'Monetary']]
y = df['Churned']

In [61]:
X.shape

(5878, 2)

In [62]:
y.shape

(5878,)

In [63]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [64]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [65]:
X_train.shape

(4702, 2)

In [66]:
X_test.shape

(1176, 2)

In [67]:
lr = LogisticRegression()

In [68]:
lr.fit(X_train, y_train)

LogisticRegression()

In [69]:
y_pred = lr.predict(X_test)

In [70]:
accuracy_score(y_test, y_pred) * 100

67.09183673469387

In [71]:
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n           0       0.79      0.47      0.59       589\n           1       0.62      0.87      0.73       587\n\n    accuracy                           0.67      1176\n   macro avg       0.70      0.67      0.66      1176\nweighted avg       0.70      0.67      0.66      1176\n'

In [72]:
from sklearn.ensemble import RandomForestClassifier

In [73]:
rfc = RandomForestClassifier()

In [74]:
rfc.fit(X_train, y_train)

RandomForestClassifier()

In [75]:
y_pred_rfc = rfc.predict(X_test)

In [76]:
accuracy_score(y_test, y_pred_rfc) * 100

61.47959183673469

In [77]:
classification_report(y_test, y_pred_rfc)

'              precision    recall  f1-score   support\n\n           0       0.63      0.57      0.60       589\n           1       0.60      0.66      0.63       587\n\n    accuracy                           0.61      1176\n   macro avg       0.62      0.61      0.61      1176\nweighted avg       0.62      0.61      0.61      1176\n'

In [78]:
rfc.score(X_train, y_train) * 100

99.76605699702255

In [79]:
rfc.score(X_test, y_test) * 100

61.47959183673469

In [80]:
rf_tuned = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=20,
    random_state=42
)

In [81]:
rf_tuned.fit(X_train, y_train)


RandomForestClassifier(max_depth=4, min_samples_leaf=20, random_state=42)

In [82]:
rf_tuned.score(X_train, y_train) * 100

70.41684389621437

In [83]:
rf_tuned.score(X_test, y_test) * 100

68.70748299319727

In [99]:
df_sales['Customer ID'] = df_sales['Customer ID'].astype(str)
df['Customer ID'] = df['Customer ID'].astype(str)

In [85]:
snapshot_date = df_sales['InvoiceDate'].max() + pd.Timedelta(days=1)

In [100]:
df['AvgOrderValue'] = df['Monetary'] / df['Frequency']

In [103]:
unique_products = df_sales.groupby('Customer ID')['StockCode'].nunique()
df['UniqueProducts'] = df['Customer ID'].map(unique_products)

In [104]:
first_purchase = df_sales.groupby('Customer ID')['InvoiceDate'].min()
tenure_series = (snapshot_date - first_purchase).dt.days
df['TenureDays'] = df['Customer ID'].map(tenure_series)

In [105]:
df.head()

,Customer ID,Recency,Frequency,Monetary,Cluster,Segment,Churned,AvgOrderValue,UniqueProducts,TenureDays
0,12346,326,12,77556.46,1,Regular,1,6463.038333,27,726
1,12347,2,8,5633.32,1,Regular,0,704.165000,126,404
2,12348,75,5,2019.40,1,Regular,0,403.880000,25,438
3,12349,19,4,4428.69,1,Regular,0,1107.172500,138,589
4,12350,310,1,334.40,0,At Risk,1,334.400000,17,310


In [106]:
df.isnull().sum()

,0
Customer ID,0
Recency,0
Frequency,0
Monetary,0
Cluster,0
Segment,0
Churned,0
AvgOrderValue,0
UniqueProducts,0
TenureDays,0


In [107]:
X = df[['Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'TenureDays']]
y = df['Churned']

In [108]:
df[['AvgOrderValue', 'UniqueProducts', 'TenureDays']].isna().sum()

,0
AvgOrderValue,0
UniqueProducts,0
TenureDays,0


In [109]:
print(df.index.dtype)
print(unique_products.index.dtype)

object
object


In [110]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [111]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

In [112]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(max_depth=4, min_samples_leaf=20, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(max_depth=3, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB()
}

In [113]:
results = []

In [114]:
for name, clf in models.items():
    clf.fit(X_train, y_train)
    train_acc = clf.score(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    results.append({'Model': name, 'Train Accuracy': train_acc, 'Test Accuracy': test_acc})

In [115]:
results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print(results_df)

                 Model  Train Accuracy  Test Accuracy
3    Gradient Boosting        0.845810       0.821429
2        Random Forest        0.827946       0.819728
4                  KNN        0.855168       0.811224
1        Decision Tree        0.815185       0.810374
0  Logistic Regression        0.793067       0.808673
5          Naive Bayes        0.613143       0.598639


In [116]:
final_model = RandomForestClassifier(max_depth=4, min_samples_leaf=20, random_state=42)
final_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=4, min_samples_leaf=20, random_state=42)

In [118]:
accuracy_score(y_test, final_model.predict(X_test)) * 100

81.97278911564626

In [121]:
import os
import joblib

In [122]:
joblib.dump(final_model, 'churn_model.pkl')

['churn_model.pkl']